# 03 · Filter & Rank — the shared multi-layer binder filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 07** we run the **binder** cutoffs on the pool and report honest survival (D3 pt 1).

Run `00`–`02` first so `results/designs.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

In [ ]:
import filtering_pipeline as fp
import pandas as pd
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

## Build `fp.Design` objects from the campaign

Map each binder onto the shared `Design` record (`design_type="binder"`). Layer 2 (orthogonal) needs a
*second* predictor (ESMFold/Boltz) — we run layers **(1, 3)** here and note that L2 is added once you
wire in a second predictor in a real run.

In [ ]:
df = pd.read_csv("results/designs.csv")
designs = [fp.Design(design_id=str(r.design_id), sequence=str(r.sequence), design_type="binder",
                     plddt=r.plddt, pae_interaction=r.pae_interaction, scrmsd=r.scrmsd,
                     shape_complementarity=r.shape_complementarity,
                     extra={"paradigm": r.paradigm, "synthetic": True})
           for r in df.itertuples()]
print(len(designs), "Design objects built (design_type='binder')")

## Run the pipeline + report

In [ ]:
ranked = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(ranked, top_n=10, save_prefix="results/proj07")
print("\nNOTE: numbers are SYNTHETIC (mock). Layer 2 (orthogonal ESMFold/Boltz) is added in a real run.")
top

## Survival-at-each-layer (honest accounting)
Report N pass / N generated at each layer, per paradigm — the honest hit-rate view.

In [ ]:
import pandas as pd
if "layers_passed" in ranked:
    print(ranked["layers_passed"].value_counts().sort_index())

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (`design_type='binder'`).
- [ ] Survival-at-each-layer reported (honest hit rate).
- [ ] Mapping assumptions written down.

**Next:** `04_validate.ipynb` — cross-variant breadth + head-to-head.